# Notebook 01: Escalation Pattern

The CCA exam tests whether you understand that escalation routing must use **deterministic business rules enforced in code**, not self-reported LLM confidence scores. This notebook shows both patterns on the same scenario — a \$600 refund for customer C003 — so you can see the difference in observable behavior.

Pattern covered: **PostToolUse callbacks with deterministic escalation rules** vs. confidence-based routing in the system prompt.

## Setup

In [ ]:
import sys
from pathlib import Path

# Add project root so notebooks.helpers and customer_service are importable
sys.path.insert(0, str(Path(".").resolve()))

import anthropic
from helpers import compare_results, print_usage

from customer_service.data.customers import CUSTOMERS
from customer_service.data.scenarios import SCENARIOS
from customer_service.services.audit_log import AuditLog
from customer_service.services.container import ServiceContainer
from customer_service.services.customer_db import CustomerDatabase
from customer_service.services.escalation_queue import EscalationQueue
from customer_service.services.financial_system import FinancialSystem
from customer_service.services.policy_engine import PolicyEngine

In [ ]:
def make_services() -> ServiceContainer:
    """Create a fresh ServiceContainer with seed customer data."""
    return ServiceContainer(
        customer_db=CustomerDatabase(CUSTOMERS),
        policy_engine=PolicyEngine(),
        financial_system=FinancialSystem(),
        escalation_queue=EscalationQueue(),
        audit_log=AuditLog(),
    )


client = anthropic.Anthropic()
scenario = SCENARIOS["amount_threshold"]  # C003, $600 refund
print(f"Scenario: {scenario['customer_id']} - {scenario['message']}")
print(f"Expected outcome: {scenario['expected_outcome']}")

## Anti-Pattern: LLM Confidence-Based Escalation

<div style="border-left: 4px solid #dc3545; padding: 12px 16px; background: #fff5f5; margin: 8px 0;">
<strong>What's wrong:</strong> The system prompt asks Claude to self-rate confidence from 0–100 and only escalate if below 70. Claude typically rates itself highly confident — even on a $600 refund that should trigger mandatory escalation — and handles the case directly instead of escalating to a human agent.
</div>

In [ ]:
from customer_service.anti_patterns.confidence_escalation import run_confidence_agent

# Fresh services to isolate anti-pattern run
anti_services = make_services()
print("Running anti-pattern (confidence-based escalation)...")
anti_result = run_confidence_agent(client, anti_services, scenario["message"])
print(f"Stop reason: {anti_result.stop_reason}")
print(f"Tool calls: {[tc['name'] for tc in anti_result.tool_calls]}")
print(f"\nAgent response:\n{anti_result.final_text[:500]}")

In [ ]:
# THE KEY CHECK: did the anti-pattern actually escalate?
anti_escalations = anti_services.escalation_queue.get_queue()
print(f"Escalation queue length: {len(anti_escalations)}")
if not anti_escalations:
    print("FAILURE: Agent did NOT escalate despite $600 refund (amount > $500 threshold)")
    print("The confidence-based approach let Claude handle the case itself.")
else:
    print("Note: Agent did escalate this run (probabilistic behavior varies by run)")
    for esc in anti_escalations:
        print(f"  Escalation reason: {esc.reason}")

In [ ]:
# Token usage for anti-pattern run
# AgentResult.usage is a UsageSummary dataclass — wrap it for print_usage
class _UsageWrapper:
    def __init__(self, u):
        self.usage = u


print_usage(_UsageWrapper(anti_result.usage))

## Correct Pattern: Deterministic Callback Enforcement

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Why this works:</strong> PostToolUse callbacks check business rules in code: amount &gt; $500 triggers mandatory escalation. The callback blocks the <code>process_refund</code> tool call and returns a result telling Claude it must call <code>escalate_to_human</code>. This is deterministic — Claude cannot talk its way out of escalation.
</div>

In [ ]:
from customer_service.agent import build_callbacks, get_system_prompt, run_agent_loop

correct_services = make_services()
callbacks = build_callbacks()
print("Running correct pattern (deterministic callback enforcement)...")
correct_result = run_agent_loop(
    client,
    correct_services,
    scenario["message"],
    get_system_prompt(),
    callbacks=callbacks,
)
print(f"Stop reason: {correct_result.stop_reason}")
print(f"Tool calls: {[tc['name'] for tc in correct_result.tool_calls]}")
print(f"\nAgent response:\n{correct_result.final_text[:500]}")

In [ ]:
# Verify: correct pattern MUST escalate the $600 refund
correct_escalations = correct_services.escalation_queue.get_queue()
print(f"Escalation queue length: {len(correct_escalations)}")
if correct_escalations:
    print("SUCCESS: Agent correctly escalated the $600 refund case")
    esc = correct_escalations[0]
    print(f"  Customer: {esc.customer_id}")
    print(f"  Reason: {esc.reason}")
else:
    print("UNEXPECTED: Correct pattern did not escalate — check callbacks.py")

In [ ]:
print_usage(_UsageWrapper(correct_result.usage))

## Compare Results

In [ ]:
compare_results(
    {
        "escalated": bool(anti_escalations),
        "escalation_queue_size": len(anti_escalations),
        "tool_calls": len(anti_result.tool_calls),
    },
    {
        "escalated": bool(correct_escalations),
        "escalation_queue_size": len(correct_escalations),
        "tool_calls": len(correct_result.tool_calls),
    },
)

> **CCA Exam Tip:** If an exam answer mentions "confidence threshold," "self-assessment," or "let the model decide when to escalate," it is ALWAYS wrong. Escalation routing must use deterministic business rules enforced in code (PostToolUse callbacks), never LLM self-reported confidence. The correct answer always involves programmatic enforcement.

## Summary

- **Anti-pattern failure:** Confidence-based routing in the system prompt is probabilistic — Claude self-reports high confidence and skips escalation for a \$600 refund that business rules require to be escalated.
- **Correct pattern guarantee:** PostToolUse callbacks enforce deterministic rules: `amount > 500` blocks `process_refund` and forces Claude to call `escalate_to_human`, regardless of what Claude thinks its confidence is.
- **Key principle:** System prompts provide context; callbacks enforce rules. This is the CCA #1 architectural principle — programmatic enforcement beats prompt-based guidance every time.